# ECE 491 — Python Lab 4
## MNIST Digit Recognition Using a Shallow Neural Network
### Complete, carefully reviewed Google Colab | Questions 1–7

**Source:** *Python Lab 4 — ECE 491*, three-page professor handout.

This notebook follows the professor's sequence and solves **every question from Q1 to Q7**. For each question, it includes the original task, the necessary theory, numbered mini-steps, executable code, the result/interpretation, and verification where possible. A substantial theory cheat sheet appears at the end.

**Professor's core requirements**

- Use **TensorFlow / Keras** and the **MNIST handwritten-digit dataset**, consisting of 60,000 training and 10,000 test images.
- Create an **80% / 20% training–validation split** from the original training set.
- Train the professor's original shallow model: `Flatten → Dense(128, ReLU) → Dropout(0.2) → Dense(10)`.
- Use `SparseCategoricalCrossentropy(from_logits=True)`, the **Adam** optimizer, and **5 epochs**.
- Report **validation** and **test** accuracy.
- Train a **second** model: `Flatten → Dense(200, ReLU) → Dense(200, ReLU) → Dense(10)` for **5 epochs**. Compare both accuracy values.

> **Source alignment:** Unlike the separate Homework 4 task, *this Lab 4 explicitly permits* `tf.keras.datasets.mnist.load_data()`; it is therefore used here as printed in the handout. No restrictions from a different assignment are imported into this lab.

**Accuracy integrity:** Q5, Q6 and Q7 demand measurements from actual training. Their numerical results are calculated automatically when you run this notebook. They are **not pre-filled with guessed scores**. TensorFlow version, hardware and randomness can cause small result variations.

**Math style:** All displayed equations use double-dollar delimiters.

## 0. Mathematical overview — what are we building?

An MNIST image is a grey-scale matrix of **28 × 28** pixels. Every pixel initially has an integer intensity from 0 (black) to 255 (white). The first layer flattens the image into a vector of length

$$
28\times 28=784.
$$

A neural network then learns to map those pixel intensities to one of ten digits:

$$
y\in\{0,1,2,3,4,5,6,7,8,9\}.
$$

The core operation of a Dense layer is

$$
\boxed{\mathbf{z}=\mathbf{W}\mathbf{x}+\mathbf{b}}.
$$

Hidden layers use **ReLU**:

$$
\boxed{\operatorname{ReLU}(z)=\max(0,z)}.
$$

The output layer in the professor's model returns ten **logits** (unrestricted class scores). To classify an image, choose the index of the largest logit:

$$
\boxed{\hat y=\underset{k\in\{0,\ldots,9\}}{\arg\max}\;z_k}.
$$

**Simple analogy:** The 784 input pixels are observations, the Dense layers learn combinations of features, and the ten final scores act like ten competing votes for the digits 0–9.

## Setup — Imports and reproducibility

Run this cell first. TensorFlow is available in the standard Google Colab runtime. The seed makes the comparison more repeatable, although some hardware operations may still vary.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 491
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

# Question 1 — Print the shapes of the four MNIST variables

### Professor's task

Print the shapes of `x_train`, `y_train`, `x_test`, and `y_test`.

### Step 1 — Load the original dataset

The handout explicitly permits:

```python
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
```

### Step 2 — Understand the four outputs

| Variable | Meaning | Expected shape |
|---|---|---|
| `x_train` | 60,000 images, each 28 × 28 | `(60000, 28, 28)` |
| `y_train` | One integer digit label per training image | `(60000,)` |
| `x_test` | 10,000 independent images | `(10000, 28, 28)` |
| `y_test` | One integer digit label per test image | `(10000,)` |

Mathematically, the complete training image array belongs to

$$
\boxed{X_{\mathrm{train}}\in\mathbb{R}^{60000\times28\times28}}
$$

and the test images belong to

$$
\boxed{X_{\mathrm{test}}\in\mathbb{R}^{10000\times28\times28}}.
$$

### Step 3 — Normalize the images

Follow the professor's formula:

$$
\boxed{x_{\mathrm{norm}}=\frac{x_{\mathrm{raw}}}{255}}.
$$

This changes the *values* to the interval $[0,1]$, but does not change the array shapes.

In [ ]:
# Q1: Load MNIST exactly as permitted in this Lab 4.
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Print the original four shapes BEFORE splitting for Question 2.
print("Q1 — x_train shape:", x_train_full.shape)
print("Q1 — y_train shape:", y_train_full.shape)
print("Q1 — x_test  shape:", x_test.shape)
print("Q1 — y_test  shape:", y_test.shape)

assert x_train_full.shape == (60000, 28, 28)
assert y_train_full.shape == (60000,)
assert x_test.shape == (10000, 28, 28)
assert y_test.shape == (10000,)

# 0–255 unsigned integers -> 0–1 floating point values.
x_train_full = x_train_full.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

print("Pixel range after normalization:", float(x_train_full.min()), "to", float(x_train_full.max()))
print("Q1 checks: PASS")

### Q1 — Final answer

$$
\boxed{\begin{aligned}
\operatorname{shape}(x_{\mathrm{train}})&=(60000,28,28),\\
\operatorname{shape}(y_{\mathrm{train}})&=(60000,),\\
\operatorname{shape}(x_{\mathrm{test}})&=(10000,28,28),\\
\operatorname{shape}(y_{\mathrm{test}})&=(10000,).
\end{aligned}}
$$

The number of samples is the first dimension. Each image contains $784$ pixel values when flattened.

# Question 2 — Split the original training data into 80% training and 20% validation

### Professor's task

Split the **60,000** original training examples into training and validation subsets.

### Mini-step 1 — Calculate the sizes

$$
N_{\mathrm{train}}=0.80\times60000=\boxed{48000}
$$

$$
N_{\mathrm{val}}=0.20\times60000=\boxed{12000}.
$$

The separate **10,000-example test set must not be included in this split**.

### Mini-step 2 — Use one reproducible partition

A random permutation with a fixed seed gives one explicit 80/20 split. The same `x_train`, `y_train`, `x_val`, `y_val` arrays will be used for **both Q5 and Q7**, ensuring the two models are compared on the same data.

### Mini-step 3 — Verify matching images and labels

Every image must remain paired with its correct digit label, so **the same indices** are used to select both images and labels.

In [ ]:
# Q2: 80% training, 20% validation from the original 60,000 examples.
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(y_train_full))

split = int(0.80 * len(indices))
train_indices = indices[:split]
val_indices = indices[split:]

x_train = x_train_full[train_indices]
y_train = y_train_full[train_indices]
x_val = x_train_full[val_indices]
y_val = y_train_full[val_indices]

print("Q2 — x_train:", x_train.shape)
print("Q2 — y_train:", y_train.shape)
print("Q2 — x_val:  ", x_val.shape)
print("Q2 — y_val:  ", y_val.shape)
print("Preserved test set:", x_test.shape)

assert len(x_train) == len(y_train) == 48000
assert len(x_val) == len(y_val) == 12000
assert len(x_test) == len(y_test) == 10000
assert len(set(train_indices) & set(val_indices)) == 0
print("Q2 checks: PASS")

### Q2 — Final answer

| Data subset | Images | Labels |
|---|---:|---|
| Training | **48,000** | `y_train` |
| Validation | **12,000** | `y_val` |
| Independent test | **10,000** | `y_test` |

**Why validation is important:** It measures progress on images that the network is not currently fitting. It is distinct from the test set, which is reserved for evaluation after training.

## Professor's original model — shared by Questions 3–6

The handout specifies this exact **sequence of functional layers**:

$$
\underbrace{28\times28}_{\text{input}}
\to\underbrace{784}_{\text{Flatten}}
\to\underbrace{128}_{\text{Dense + ReLU}}
\to\underbrace{\text{Dropout}(0.2)}_{\text{training only}}
\to\underbrace{10}_{\text{Dense logits}}.
$$

The code below uses `tf.keras.Input(shape=(28,28))` to declare the input shape explicitly. This is **functionally equivalent** to the professor's `Flatten(input_shape=(28,28))` and avoids a current-Keras style warning. The functional layers, units, activations, and dropout rate are unchanged.

In [ ]:
# Professor's original model, written using a current-Keras-compatible input declaration.
model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10)  # Ten logits: no softmax in this layer.
], name="lab4_original_model")

model.summary()
print("Total trainable parameters:", model.count_params())
assert model.count_params() == 101770

# Question 3 — Define Flatten, Dense, and Dropout, and explain every parameter

The professor asks for definitions grounded in TensorFlow documentation. The definitions below correspond to these official pages:

- [TensorFlow: Flatten](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten)
- [TensorFlow: Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)
- [TensorFlow: Dropout](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout)

## A. `Flatten(input_shape=(28,28))`

**Definition:** Converts each input image into a one-dimensional vector. It does **not** learn weights and does **not** discard information.

$$
\boxed{(28,28)\longrightarrow(784)}
$$

The parameter `input_shape=(28, 28)` means that **one image** has 28 rows and 28 columns. The batch size is separate, so a batch of $B$ images is transformed as

$$
(B,28,28)\longrightarrow(B,784).
$$

**Example:** 32 images yield an array of shape `(32,784)`.

## B. `Dense(128, activation='relu')`

**Definition:** A fully connected layer: each of its 128 output neurons receives a weighted combination of all 784 inputs, plus a bias.

$$
\boxed{\mathbf{a}=\operatorname{ReLU}(\mathbf{W}\mathbf{x}+\mathbf{b})}
$$

- `128` means **128 output neurons / units**.
- `activation='relu'` applies $\max(0,z)$ after the weighted sum.
- If `activation` were not supplied, the Dense layer would use a **linear** activation by default.
- By default, `use_bias=True`, so each neuron also learns a bias.

**Weight shape:** $784\times128$; **bias shape:** $128$. Total trainable parameters:

$$
784\times128+128=\boxed{100480}.
$$

## C. `Dropout(0.2)`

**Definition:** During each training update, the layer randomly sets a fraction of its input activations to zero. In Keras, the retained activations are appropriately scaled during training, and Dropout is inactive at inference/evaluation time.

- `0.2` is the **drop rate of 20%** per training step.
- Its purpose is to reduce reliance on particular activations and help regularize training.
- It introduces **zero trainable parameters**.
- It does **not** remove 20% of neurons from the architecture permanently.

## D. `Dense(10)`

**Definition:** Produces one real-valued score for each of the ten possible digits.

- `10` means **ten class scores (logits)**.
- No activation is specified, so the layer is **linear**.
- Using `SparseCategoricalCrossentropy(from_logits=True)` later is therefore correct.
- Parameter count:

$$
128\times10+10=\boxed{1290}.
$$

## E. Total model parameters

$$
\boxed{100480+0+1290=101770\text{ trainable parameters}}.
$$

**Why this matters:** `Flatten` changes shape, the two `Dense` layers learn parameters, and `Dropout` regularizes the training process.

In [ ]:
# Q3: Verify the type, output shape, and number of parameters for each layer.
for layer in model.layers:
    print(
        f"{layer.name:22s} | "
        f"type={type(layer).__name__:8s} | "
        f"output={str(layer.output.shape):16s} | "
        f"parameters={layer.count_params():6d}"
    )

assert sum(layer.count_params() for layer in model.layers) == 101770
print("Q3 parameter check: PASS")

# Question 4 — Print the model's output shape

The professor provides the older TensorFlow syntax:

```python
print(model.output.get_shape().as_list())
```

On more recent Keras versions, `model.output` may be a symbolic `KerasTensor` without that method. A **version-compatible** alternative is:

```python
print(list(model.output_shape))
```

### Explanation

The model emits ten logits **for each input image**.

$$
\boxed{\operatorname{output\ shape}=(\mathrm{None},10)}.
$$

- `None` means the batch size can vary.
- `10` means one output score per possible digit.
- Examples: 1 image → `(1,10)`; 32 images → `(32,10)`.

In [ ]:
# Q4: Compatible with modern Keras.
print("Q4 — output shape:", list(model.output_shape))
assert tuple(model.output_shape) == (None, 10)
print("Q4 check: PASS")

## Training mathematics used in Questions 5–7

The professor specifies:

```python
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
```

### Step 1 — Convert logits to class probabilities (conceptually)

If the model outputs logits $z_0,\ldots,z_9$, their corresponding Softmax probabilities are

$$
\boxed{p_k=\frac{e^{z_k}}{\sum_{j=0}^{9}e^{z_j}}}.
$$

### Step 2 — Compute the sparse categorical cross-entropy

For a ground-truth integer digit $y$, the loss for one image is

$$
\boxed{\mathcal L=-\log p_y}.
$$

Keras calculates this **directly and stably from logits** when `from_logits=True`. Therefore, **do not add a Softmax activation to the final Dense layer unless you also change the loss setting to `from_logits=False`.**

### Step 3 — Backpropagation and Adam

Keras automatically differentiates the loss with respect to the trainable weights, and the Adam optimizer updates those weights. Adam's default initial learning rate is **0.001** for the default configuration, although this lab simply asks us to use `optimizer='adam'`.

### Step 4 — Accuracy

$$
\boxed{\mathrm{Accuracy}=\frac{\#\text{ correctly classified images}}{\#\text{ images}}\times100\%}.
$$

**Q5 and Q7 specify exactly five epochs.** The batch size is set explicitly to 32, matching Keras's usual array-input default; both models use the same optimizer, loss, split, and batch size for a fair comparison.

# Question 5 — Train the original model for five epochs and report validation accuracy

### Mini-step 1 — Compile exactly as requested

$$
\text{optimizer}=\mathrm{Adam},\quad
\text{loss}=\mathrm{SparseCategoricalCrossentropy(from\_logits=True)}.
$$

### Mini-step 2 — Train on 48,000 images

Use `validation_data=(x_val,y_val)` so the **12,000 validation images are not used for weight updates**.

### Mini-step 3 — Record validation accuracy after every epoch

Keras stores these values in `history.history['val_accuracy']`.

**Important:** The homework does not supply an exact target percentage. Your genuine results will appear when the cell finishes training. Do not copy an accuracy from a different run.

In [ ]:
# Q5: Compile and train the ORIGINAL model for exactly five epochs.
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer="adam",
    loss=loss_fn,
    metrics=["accuracy"]
)

history_1 = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=32,
    verbose=2
)

val_acc_1 = float(history_1.history["val_accuracy"][-1])
print("\nQ5 — Validation accuracy after each epoch:")
for i, acc in enumerate(history_1.history["val_accuracy"], start=1):
    print(f"  Epoch {i}: {acc*100:.2f}%")
print(f"Q5 — FINAL validation accuracy (Epoch 5): {val_acc_1*100:.2f}%")

assert len(history_1.history["val_accuracy"]) == 5
assert 0 <= val_acc_1 <= 1

### Q5 — Report-ready answer (generated by your run)

The original shallow network was trained for **five epochs** using Adam and sparse categorical cross-entropy. Its validation accuracy in each epoch is printed above. The final answer is the actual **Epoch 5** value, not a theoretical constant.

# Question 6 — Evaluate test accuracy of the original model

The professor supplies:

```python
model.evaluate(x_test, y_test, verbose=2)
```

### Mini-step 1 — Preserve the independent test data

Unlike the 48,000 examples used for training and 12,000 used for validation, the **10,000 test examples** have not been used to update weights.

### Mini-step 2 — Evaluate the final trained model

The method returns **test loss** and **test accuracy**.

### Mini-step 3 — Report the percentage

$$
\boxed{\mathrm{Test\ accuracy}(\%)=100\times\mathrm{model.evaluate(\ldots)[1]}}.
$$

This is an experimental result and must be measured from the trained model.

In [ ]:
# Q6: Official MNIST test-set evaluation.
test_loss_1, test_acc_1 = model.evaluate(x_test, y_test, verbose=2)
test_loss_1 = float(test_loss_1)
test_acc_1 = float(test_acc_1)

print(f"Q6 — Test loss: {test_loss_1:.6f}")
print(f"Q6 — Test accuracy: {test_acc_1*100:.2f}%")
assert 0 <= test_acc_1 <= 1

### Q6 — Final answer

Report the **measured test accuracy** displayed by the cell above. It is not interchangeable with validation accuracy; the two metrics come from different datasets.

# Question 7 — Build a new 200–200 neural network; train five epochs; compare accuracies

### Professor's architecture

> Input layer → Dense layer (output size = 200) → Dense layer (output size = 200) → output layer. Use ReLU as the activation function.

### Mini-step 1 — Interpret the classification output correctly

The two **hidden Dense layers** use ReLU. The last Dense layer has ten **linear logits**, as in the original model, so the professor's `from_logits=True` cross-entropy remains valid. Applying ReLU to the final logits would not be appropriate for this same setup.

$$
\boxed{28\times28\to784\to200\ \mathrm{ReLU}\to200\ \mathrm{ReLU}\to10\ \mathrm{logits}}.
$$

The professor specifies no Dropout for this new architecture, so none is added.

### Mini-step 2 — Count the parameters

First Dense layer:

$$
(784\times200)+200=\boxed{157000}.
$$

Second Dense layer:

$$
(200\times200)+200=\boxed{40200}.
$$

Final Dense layer:

$$
(200\times10)+10=\boxed{2010}.
$$

Total:

$$
\boxed{157000+40200+2010=199210\ \text{trainable parameters}}.
$$

### Mini-step 3 — Reuse the SAME training and validation sets

Train for **five epochs** with Adam, sparse categorical cross-entropy (logits), and batch size 32.

### Mini-step 4 — Record validation and test accuracy

Use `history_2.history['val_accuracy']` for Q7's validation metric and `model_2.evaluate(...)` for test accuracy. The numerical comparison is performed automatically below.

In [ ]:
# Q7: SECOND model with TWO hidden Dense layers of 200 units each.
model_2 = tf.keras.models.Sequential([
    tf.keras.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(10)  # Raw logits; matches from_logits=True.
], name="lab4_200_200_model")

model_2.summary()
print("Q7 — Parameters:", model_2.count_params())
assert model_2.count_params() == 199210
assert tuple(model_2.output_shape) == (None, 10)

model_2.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_2 = model_2.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=5,
    batch_size=32,
    verbose=2
)

val_acc_2 = float(history_2.history["val_accuracy"][-1])
test_loss_2, test_acc_2 = model_2.evaluate(x_test, y_test, verbose=2)
test_loss_2 = float(test_loss_2)
test_acc_2 = float(test_acc_2)

print("\nQ7 — Second network validation accuracy by epoch:")
for i, acc in enumerate(history_2.history["val_accuracy"], start=1):
    print(f"  Epoch {i}: {acc*100:.2f}%")
print(f"Q7 — Final validation accuracy: {val_acc_2*100:.2f}%")
print(f"Q7 — Test accuracy: {test_acc_2*100:.2f}%")

assert len(history_2.history["val_accuracy"]) == 5
assert 0 <= val_acc_2 <= 1
assert 0 <= test_acc_2 <= 1

## Q7 — Direct comparison of the two models

The professor asks you to report the validation and test accuracy of the new model. For clarity, compare **both** models on **exactly the same validation and test sets**.

Define

$$
\Delta_{\mathrm{test}}=100(\mathrm{Acc}_{\mathrm{new}}-\mathrm{Acc}_{\mathrm{original}})
$$

in **percentage points**.

A deeper/wider network *may* learn more complex patterns, but **a larger network does not guarantee higher test accuracy**. Data, capacity, training duration and overfitting all influence performance. Use your actual observed comparison.

In [ ]:
# Q7: Dynamic, reproducible answer table using actual training results.
comparison = [
    ("Original 128 + Dropout 0.2", 101770, val_acc_1, test_acc_1),
    ("New 200–200 ReLU", 199210, val_acc_2, test_acc_2),
]
print(f"{'Model':30s} {'Params':>9s} {'Val Acc':>12s} {'Test Acc':>12s}")
print("-" * 70)
for label, params, val_accuracy, test_accuracy in comparison:
    print(f"{label:30s} {params:9,d} {val_accuracy*100:11.2f}% {test_accuracy*100:11.2f}%")

delta_val = 100 * (val_acc_2 - val_acc_1)
delta_test = 100 * (test_acc_2 - test_acc_1)
print(f"\nChange in validation accuracy: {delta_val:+.2f} percentage points")
print(f"Change in test accuracy:       {delta_test:+.2f} percentage points")

if delta_test > 0:
    print("Measured Q7 conclusion: The 200–200 model performed better on this test run.")
elif delta_test < 0:
    print("Measured Q7 conclusion: The original model performed better on this test run.")
else:
    print("Measured Q7 conclusion: The two test accuracies were equal.")

In [ ]:
# Optional Q5/Q7 visualization: validation accuracy across the five epochs.
epochs = np.arange(1, 6)
plt.figure(figsize=(8.8, 4.9))
plt.plot(epochs, np.asarray(history_1.history["val_accuracy"]) * 100, "o-", label="Original 128 + Dropout")
plt.plot(epochs, np.asarray(history_2.history["val_accuracy"]) * 100, "s-", label="New 200–200 ReLU")
plt.xticks(epochs)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy (%)")
plt.title("ECE 491 Lab 4 — Validation Accuracy Comparison")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Submission-ready final answer sheet — Q1 to Q7

**The following cell is designed to be copied into your lab report after running the complete notebook.** It prints all fixed answers and dynamically inserts the genuine experimental validation/test accuracy results rather than using invented values.

In [ ]:
print("="*66)
print("ECE 491 — PYTHON LAB 4: FINAL ANSWERS")
print("="*66)
print("Q1 — x_train: (60000, 28, 28); y_train: (60000,)")
print("     x_test: (10000, 28, 28); y_test: (10000,)")
print("Q2 — 48,000 training images; 12,000 validation images.")
print("Q3 — Flatten: (28,28)->784; Dense(128, ReLU): 128 neurons;")
print("     Dropout(0.2): drops 20% of activations during training;")
print("     Dense(10): 10 logits, no output activation.")
print("Q4 — Model output shape: [None, 10].")
print(f"Q5 — Original model final validation accuracy: {val_acc_1*100:.2f}%.")
print(f"Q6 — Original model test accuracy: {test_acc_1*100:.2f}%.")
print("Q7 — New network: 784->200(ReLU)->200(ReLU)->10(logits).")
print(f"     Validation accuracy: {val_acc_2*100:.2f}%.")
print(f"     Test accuracy: {test_acc_2*100:.2f}%.")
print(f"     Test-accuracy change: {delta_test:+.2f} percentage points.")
print("="*66)

# Final theory cheat sheet — ECE 491 Lab 4

## A. Dataset and partitioning

| Term | Meaning | This lab |
|---|---|---|
| MNIST | Database of handwritten digit images | 60,000 train + 10,000 test |
| Image | Single 28 × 28 grayscale picture | 784 pixels |
| Normalization | Map pixel intensity from 0–255 to 0–1 | `x/255.0` |
| Training set | Examples used for gradient updates | 48,000 images |
| Validation set | Unseen-during-update images for training assessment | 12,000 images |
| Test set | Reserved for final generalization evaluation | 10,000 images |

## B. Layers

| Layer | Mathematical effect | Learns parameters? |
|---|---|---|
| Flatten | $(28,28)\to784$ | No |
| Dense | $\mathbf z=\mathbf W\mathbf x+\mathbf b$ | Yes |
| ReLU | $\max(0,z)$ | No |
| Dropout(0.2) | Masks ~20% of activations *during training* | No |
| Dense(10) | Ten digit class logits | Yes |

## C. Why we use raw logits

The teacher's final layer is `Dense(10)`, not `Dense(10, activation='softmax')`.

Therefore, the matching loss is

$$
\boxed{\mathrm{SparseCategoricalCrossentropy(from\_logits=True)}}.
$$

Softmax is handled internally by the loss's stable computation.

## D. Parameter counting

Dense-layer parameter formula:

$$
\boxed{N_{\mathrm{params}}=N_{\mathrm{inputs}}N_{\mathrm{units}}+N_{\mathrm{units}}}.
$$

- Original 128-hidden model:

$$
\boxed{784(128)+128+128(10)+10=101770}.
$$

- Two-hidden-layer 200–200 model:

$$
\boxed{784(200)+200+200(200)+200+200(10)+10=199210}.
$$

## E. Core formulas

**Dense layer:**

$$
\boxed{\mathbf z=\mathbf W\mathbf x+\mathbf b}.
$$

**ReLU activation:**

$$
\boxed{a(z)=\max(0,z)}.
$$

**Softmax probabilities:**

$$
\boxed{p_k=\frac{e^{z_k}}{\sum_j e^{z_j}}}.
$$

**Cross-entropy for integer target $y$:**

$$
\boxed{\mathcal L=-\log p_y}.
$$

**Classification:**

$$
\boxed{\hat y=\arg\max_k z_k}.
$$

**Accuracy:**

$$
\boxed{\mathrm{Acc}=\frac{\#\mathrm{correct}}{\#\mathrm{total}}}.
$$

## F. Common grading errors

1. **Do not train on all 60,000 images** after splitting: the 12,000 validation images must be held out.
2. **Do not split the official 10,000 test images** into training/validation groups.
3. **Do not report guessed validation/test accuracy**: record actual five-epoch results.
4. **Do not mix raw logits with `from_logits=False`** or probability outputs with `from_logits=True`.
5. **Do not confuse 20% Dropout with a permanent reduction of the number of neurons.**
6. **Do not assume the two networks must have equal accuracy**, or that the larger network necessarily wins.
7. In modern Keras, `list(model.output_shape)` reliably gives the professor's intended output-shape result.

## G. Two-minute mental model

$$
\boxed{\text{Image}\rightarrow\text{Flatten}\rightarrow\text{Dense/ReLU}\rightarrow\text{Logits}\rightarrow\text{Loss}}.
$$

During training, Keras automatically performs backpropagation and Adam updates. After each epoch, validation measures performance on held-out training images. At the end, evaluate the unseen test set and compare both architectures.

---

### Official reference links

- [TensorFlow beginner tutorial](https://www.tensorflow.org/tutorials/quickstart/beginner)
- [TensorFlow Flatten](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten)
- [TensorFlow Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)
- [TensorFlow Dropout](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout)

# Final validation / integrity checks

These checks ensure the solutions are internally aligned with the professor's required architecture, split, training duration and experimental results.

In [ ]:
# Confirm no split leakage, correct shapes, five epochs, and measured accuracies.
assert x_train.shape == (48000, 28, 28)
assert x_val.shape == (12000, 28, 28)
assert x_test.shape == (10000, 28, 28)
assert list(model.output_shape) == [None, 10]
assert list(model_2.output_shape) == [None, 10]
assert model.count_params() == 101770
assert model_2.count_params() == 199210
assert len(history_1.epoch) == 5
assert len(history_2.epoch) == 5
assert np.isfinite([val_acc_1, test_acc_1, val_acc_2, test_acc_2]).all()
assert 0 <= min(val_acc_1, test_acc_1, val_acc_2, test_acc_2)
assert max(val_acc_1, test_acc_1, val_acc_2, test_acc_2) <= 1
print("PASS — All fixed mathematics, network dimensions, training durations, and measured metrics are consistent.")